# 🎙️ ROTBOTS — The Voice

---

Narration for **all** scenes (AI + found footage). The voice runs over everything.

> **🤔** What does a synthetic voice do to trust?

---

In [ ]:
# SETUP
!pip install -q edge-tts
import json, subprocess, time as _time, edge_tts
from pathlib import Path
from IPython.display import display, Markdown, Audio, HTML
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
def progress_bar(c,t,l='',w=30):
    p=c/t if t>0 else 0;f=int(w*p);return f'<div style="font-family:monospace;font-size:14px;padding:2px 0;">{"█"*f}{"░"*(w-f)} {c}/{t} {l} ({p:.0%})</div>'
def progress_html(title,c,t,l='',d='',color='#4ecca3'):
    return f'<div style="background:#1a1a2e;padding:12px;border-radius:8px;color:#eaeaea;font-family:monospace;"><div style="font-size:16px;font-weight:bold;color:{color};">{title}</div>{progress_bar(c,t,l)}'+(f'<div style="color:#a0a0a0;font-size:12px;margin-top:4px;">{d}</div>' if d else '')+'</div>'
print('✅ Setup')

In [ ]:
# LOAD SESSION
sessions = sorted([d.name for d in BASE_DIR.iterdir() if d.is_dir() and (d/'video_plan.json').exists()])
SESSION_NAME = sessions[-1] if sessions else ''
SESSION_DIR = BASE_DIR / SESSION_NAME
with open(SESSION_DIR/'video_plan.json') as f: plan = json.load(f)
print(f'✅ Session: {SESSION_NAME}')

In [ ]:
# LOAD ESSAY NARRATION
AUDIO_DIR = SESSION_DIR / 'audio'; AUDIO_DIR.mkdir(exist_ok=True)
essay_file = SESSION_DIR / 'essay_script.json'
if essay_file.exists():
    with open(essay_file) as f: essay = json.load(f)
    narration_text = essay.get('narration', '')
    if narration_text:
        wc = len(narration_text.split())
        print(f'\ud83d\udcd6 Full essay narration: {wc} words ~ {wc/2.5:.0f}s')
    else:
        print('\u274c No narration field in essay_script.json!')
else:
    print('\u274c No essay_script.json!')
    narration_text = ''


In [ ]:
# VOICE
VOICES={'female_us':'en-US-AriaNeural','male_us':'en-US-GuyNeural','female_uk':'en-GB-SoniaNeural','male_uk':'en-GB-RyanNeural','documentary':'en-US-GuyNeural','dramatic':'en-GB-RyanNeural','energetic':'en-US-JennyNeural'}
CHOSEN_VOICE='documentary'
voice_name=VOICES[CHOSEN_VOICE]
print(f'🎙️ {CHOSEN_VOICE} ({voice_name})')

In [ ]:
# GENERATE SINGLE NARRATION
async def gen_tts(text, path, voice, rate='+0%'):
    await edge_tts.Communicate(text, voice, rate=rate).save(str(path))
def get_dur(path):
    try: return float(subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(path)],capture_output=True,text=True,timeout=10).stdout.strip())
    except: return 5.0

prog = display(HTML(''), display_id=True)
t0 = _time.time()
out = AUDIO_DIR / 'narration_full.mp3'
audio_file = None

if narration_text:
    prog.update(HTML(progress_html('\ud83c\udfb5 Generating full narration...', 0, 1, 'TTS')))
    try:
        await gen_tts(narration_text, out, voice_name)
        audio_duration = get_dur(out)
        audio_file = {'path': str(out), 'duration': audio_duration}
        print(f'   narration_full.mp3: {audio_duration:.1f}s')
    except Exception as e:
        print(f'   \u274c {e}')
else:
    print('No narration text to generate!')

manifest = {'voice': voice_name, 'file': audio_file}
with open(SESSION_DIR / 'audio_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

if audio_file:
    prog.update(HTML(progress_html(f'\u2705 Narration: {audio_file["duration"]:.1f}s ({_time.time()-t0:.1f}s)', 1, 1, 'TTS')))


In [ ]:
# PREVIEW
if audio_file and Path(audio_file['path']).exists():
    display(Markdown(f'### Full Narration ({audio_file["duration"]:.1f}s)'))
    display(Audio(audio_file['path'], autoplay=False))
else:
    print('No audio to preview')


---
*ROTBOTS Workshop — LI-MA 2026*